In [1]:
from Util.Problems import Problem, solution
import math

class P009(Problem):
    number = 9
    title = "Special Pythagorean Triplet"
    description = """<p>A Pythagorean triplet is a set of three natural numbers, $a \\lt b \\lt c$, for which,
$$a^2 + b^2 = c^2.$$</p><p>For example, $3^2 + 4^2 = 9 + 16 = 25 = 5^2$.</p><p>There exists exactly one Pythagorean triplet for which $a + b + c = 1000$.<br/>Find the product $abc$.</p>"""
    abc_sum = 1000

In [2]:
p = P009()
p.describe()

## Problem 9: Special Pythagorean Triplet

<p>A Pythagorean triplet is a set of three natural numbers, $a \lt b \lt c$, for which,
$$a^2 + b^2 = c^2.$$</p><p>For example, $3^2 + 4^2 = 9 + 16 = 25 = 5^2$.</p><p>There exists exactly one Pythagorean triplet for which $a + b + c = 1000$.<br/>Find the product $abc$.</p>

### Solution notes
First we loop through all $a, b, c$ in the range $1-1000$, multiply them and if we happen to find the solution, we return it.

In [3]:
@solution(P009, max_tests= 10, first=True, make_fast=True, warmup_args=(P009.abc_sum,))
def brute_force(abc_sum):
    for a in range(1, 1000):
        for b in range(1, 1000):
            for c in range(1, 1000):
                if a ** 2 + b ** 2 == c ** 2:
                    if a + b + c == abc_sum:
                        return a * b * c
    return None

In [4]:
p.test_all(repeats=1)

31875000 found after 10 tests in 42.141750 ms by brute_force (first)


Then we do the same thing but with some optimisations, ensuring $b > a$, aborting the loop if $a + 2b + 1 > 1000$, as this is the smalles $a + b + c$ can be, so if it is already $>1000$, this can never be the result. Finally, the inner checks of whether the sum is $1000$ and whether this is a Pythagorean triplet are reversed, so the least expensive is done first.

In [5]:
@solution(P009, max_tests= 10, make_fast=True, warmup_args=(P009.abc_sum,))
def optimised_brute_force(abc_sum):
    for a in range(1, 1000):
        for b in range(a + 1, 1000):
            # This is the smallest the total sum can be, if it is already >= 1000, we need not enter the inner loop
            if a + 2 * b + 1 > abc_sum:
                break
            for c in range(b + 1, 1000):
                if a + b + c == abc_sum:
                    if a ** 2 + b ** 2 == c ** 2:
                        return a * b * c
    return None

In [6]:
p.test_all(repeats=1)

31875000 found after 10 tests in 43.857610 ms by brute_force (first)
31875000 found after 10 tests in 11.202790 ms by optimised_brute_force


Euclid's formula can be used to generate Pythagorean triples. Given an arbitrary pair of integers $m$ and $n$ with $m>n>0$, the formula states that the integers:

$a=m^2-n^2, b=2mn, c=m^2+n^2$

form a Pythagorean triple. This formula generates every primitive triple, but not every triple. Every triple can be generated by the slightly altered formula:

$a=k\times(m^2-n^2), b=k\times(2mn), c=k\times(m^2+n^2)$

However, for this problem, we will simply generate the primitive triples. Checking if their sum divides cleanly into 1000 will show if the answer is a multiple of this primitive.

For these numbers, the sum is

$a + b + c = m^2 - n^2 + 2mn + m^2 + n^2 = 2m^2 + 2mn$

We can check mathematically if the answer we are looking for happens to be a primitive triple by solving the equation

$1000 = 2m^2 + 2mn \\
500 = m^2 + mn \\
\frac{500}{m} = m + n \\
n = \frac{500}{m} - m \\$

If we take $m = 20$, this gives $n = \frac{500}{20} - 20 = 25 - 20 = 5$

Using $m= 20, n= 5$ provides us with the Pythagorean triple $a=20^2-5^2, b=2 \times 20 \times 5, c=20^2+5^2 \\
a=400 - 25, b= 200, c= 400 + 25 \\
a=375, b= 200, c= 425$

And indeed, $375 + 200 + 425 = 1000$. Filling this triple in in the Pythagorean formula provides:

$375^2 + 200^2 = 425^2 \\
140625 + 40000 = 180625 \\
180625 = 180625$

Proving this was indeed the Pythagorean triple we were looking for! All that is left is to determine their product.

However, to make this a more general solution, we can use the fact that $m>n$ and $n = \frac{500}{m} - m $, therefore $m > \frac{500}{m} - m $

Turning this into an equality results in:

$m = \frac{500}{m} - m \\
2m = \frac{500}{m} \\
2m^2 = 500 \\
m^2 = 250 \\
m = \sqrt{250} $

This means we only need to check integer values for which $m >= \sqrt{\frac{abc\_sum}{4}}$

In [7]:
@solution(P009, best=True, make_fast=True, warmup_args=(P009.abc_sum,))
def euclids_formula(abc_sum):
    a = b = c = 0
    for m in range(int(math.sqrt(abc_sum/4)), int(math.sqrt(abc_sum))):
        if 500 % m != 0:
            continue
        n = 500 // m - m
        if n <= 0  or n > m:
            continue
        a = m ** 2 - n ** 2
        b = 2 * m * n
        c = m ** 2 + n ** 2
    if a == 0 or b == 0 or c == 0:
        return None
    return a * b * c

In [8]:
p.test_all()

31875000 found after 10 tests in 43.389980 ms by brute_force (first)
31875000 found after 1000 tests in 0.000103 ms by euclids_formula (best)
31875000 found after 10 tests in 10.963090 ms by optimised_brute_force
